In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

data = [("James", "Sales", 3000), \
    ("Michael", "Sales", 4600), \
    ("Robert", "Sales", 4100), \
    ("Maria", "Finance", 3000), \
    ("James", "Sales", 3000), \
    ("Scott", "Finance", 3300), \
    ("Jen", "Finance", 3900), \
    ("Jeff", "Marketing", 3000), \
    ("Kumar", "Marketing", 2000), \
    ("Saif", "Sales", 4100) \
  ]
columns= ["employee_name", "department", "salary"]
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show(truncate=False)

#Distinct
distinctDF = df.distinct()
print("Distinct count: "+str(distinctDF.count()))
distinctDF.show(truncate=False)


In [0]:
distinctDF.write.mode("append").saveAsTable("workspace.sandeep.distinctDF")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
customers_df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/bronze/source_csv_files/sourcefiles/customers.csv")
products_df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/bronze/source_csv_files/sourcefiles/products.csv")
sales_df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/bronze/source_csv_files/sourcefiles/sales.csv")

In [0]:
display(customers_df)

In [0]:
display(products_df)
display(sales_df)
customers_df.write.mode("append").saveAsTable("bronze.customers")
products_df.write.mode("append").saveAsTable("bronze.products")
sales_df.write.mode("append").saveAsTable("bronze.sales")

In [0]:
from pyspark.sql.functions import sum, col
spark = SparkSession.builder.appName('salesproject').getOrCreate()

In [0]:
sales_df = spark.read\
.option("header","true")\
.option("inferSchema","true")\
.csv(f"/Volumes/workspace/bronze/source_csv_files/sourcefiles/sales.csv")

In [0]:
customers_df = spark.read\
.option("header","true")\
.option("inferSchema","true")\
.csv(f"/Volumes/workspace/bronze/source_csv_files/sourcefiles/customers.csv")
display(customers_df)

In [0]:
products_df = spark.read\
.option("header","true")\
.option("inferSchema","true")\
.csv(f"/Volumes/workspace/bronze/source_csv_files/sourcefiles/products.csv")
display(products_df)

In [0]:
sales_df=sales_df.na.drop()
display(sales_df)

In [0]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn("amount", col("price") * col("quantity"))
display(sales_df)

In [0]:
joined_df = customers_df.join(sales_df, customers_df.customer_id == sales_df.customer_id)
display(joined_df)

In [0]:
selected_df = joined_df.select(
    customers_df["customer_id"],
    customers_df["customer_name"],
    customers_df["city"],
    sales_df["product_id"],
    sales_df["amount"]
)
display(selected_df)

In [0]:
grouped_df = selected_df.groupBy("customer_id", "customer_name", "city","product_id").agg({"amount": "sum"})
display(grouped_df)

In [0]:

final_df = grouped_df.join(products_df, grouped_df.product_id == products_df.product_id)
display(final_df)

In [0]:
selected_columns_df = final_df.select(
    final_df["customer_id"],
    final_df["customer_name"],
    final_df["city"],
    products_df["product_id"],
    final_df["sum(amount)"].alias("amount"),
    products_df["product_name"]
)
display(selected_columns_df)

In [0]:
selected_columns_df.write.mode("overwrite").format("delta").saveAsTable("silver.customer_purchase_details")

In [0]:
customer_purchase_details_df = spark.read.table("silver.customer_purchase_details")
display(customer_purchase_details_df)

In [0]:
from pyspark.sql.functions import sum as _sum, collect_list, struct, size, col, when

# Group by customer_id, sum amount, and collect product_names
grouped_struct_df = selected_columns_df.groupBy("customer_id", "customer_name", "city") \
    .agg(
        _sum("amount").alias("total_amount"),
        collect_list("product_name").alias("product_names")
    ) \
    .withColumn(
        "product_names_struct",
        struct(
            col("product_names").alias("product_names")
        )
    ) \
    .select(
        "customer_id",
        "customer_name",
        "city",
        "total_amount",
        "product_names_struct"
    )

display(grouped_struct_df)

In [0]:
from pyspark.sql.functions import lit, monotonically_increasing_id, col, udf
from pyspark.sql.types import StringType

# Generate unique contact numbers with characters and numbers
def generate_contact(idx):
    chars = string.ascii_letters + string.digits
    random_part = ''.join(random.choices(chars, k=7))
    return f"{random_part}{str(idx).zfill(3)}"

generate_contact_udf = udf(generate_contact, StringType())

finalcustomerpurchasedetails = grouped_struct_df.withColumn(
    "contact_number",
    generate_contact_udf(monotonically_increasing_id())
)

# UDF to extract only numbers from contact_number
def extract_numbers(contact):
    return ''.join(filter(str.isdigit, contact))

extract_numbers_udf = udf(extract_numbers, StringType())

finalcustomerpurchasedetails = finalcustomerpurchasedetails.withColumn("extracted_numbers", extract_numbers_udf(col("contact_number")))

display(finalcustomerpurchasedetails)

In [0]:
finalcustomerpurchasedetails.write.option("mergeSchema", "true").mode("overwrite").format("delta").saveAsTable("gold.finalcustomerpurchasedetails")

In [0]:
gold_df = spark.read.table("gold.finalcustomerpurchasedetails")
display(gold_df)